# FRI-BCI: EA-EEGNet Pretraining, CRC-Hysteresis Calibration & Phase C Benchmark
**Hardware Target: Google Colab GPU (`cuda` / A100 / L4) or CPU fallback**

This notebook launches the repository's validated Stage 1 and Stage 2 pipeline (`python -m eeg_decoder.gpu_pipeline`) configured by `configs/gpu_research.json`:
- **Source pretraining (`pretrain`)**: 89 PhysioNet source subjects (`S016`–`S109` excluding known 128 Hz/anomalous IDs `88, 89, 92, 100, 104`), fitting Euclidean Alignment (EA) on `R04`, training `EEGNet` on `R04`/`R08`, and validating on `R12`.
- **Target adaptation (`adapt`)**: 15 target subjects (`S001`–`S015`: `S001`–`S010` previously inspected / development, `S011`–`S015` unopened prospective), fitting EA and adapting `EEGNet` (both `transfer` and `--scratch` controls) strictly on `R04`.
- **Target gate calibration (`calibrate`)**: Episode-level CRC-adjusted threshold selection for `HysteresisGate` strictly on `R08` (`guarantee_claimed: False`).
- **Target evaluation (`evaluate`)**: Opens held-out `R12` only when explicitly invoked.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohnclizzy/FRI-BCI-ML-RL-Decoder-Personal/blob/codex/robot-learning-design/notebooks/FRI_BCI_Colab_Master.ipynb)

In [ ]:
# [Step 1] Clone repository, install requirements-gpu.txt, and validate protocol
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/Mohnclizzy/FRI-BCI-ML-RL-Decoder-Personal.git"
BRANCH = "codex/robot-learning-design"
REPO_DIR = pathlib.Path("/content/repo") if pathlib.Path("/content").exists() else pathlib.Path.cwd()

if REPO_DIR == pathlib.Path("/content/repo"):
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-gpu.txt"], check=True)

subprocess.run([sys.executable, "-m", "eeg_decoder.gpu_pipeline", "validate", "--config", "configs/gpu_research.json"], check=True)


In [ ]:
# [Step 2] Stage 1: Source Pretraining (89 source subjects: R04/R08 train, R12 validation)
import subprocess
import sys
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK_DIR = "outputs/gpu_research"
CONFIG = "configs/gpu_research.json"
print(f"Running pretrain stage on device={DEVICE}...")

subprocess.run([
    sys.executable, "-m", "eeg_decoder.gpu_pipeline", "pretrain",
    "--config", CONFIG,
    "--work-dir", WORK_DIR,
    "--device", DEVICE,
    "--resume",
], check=True)


In [ ]:
# [Step 3] Stage 1 & 2: Target Adaptation (R04) and Episode-Level CRC Calibration (R08)
# By default, runs on exploratory/previously-inspected subjects (S001-S010).
# Keep RUN_EVALUATE_R12 = False until ready to open R12 for a target cohort.
import subprocess
import sys

DEVELOPMENT_TARGETS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
PROSPECTIVE_TARGETS = [11, 12, 13, 14, 15]
RUN_PROSPECTIVE_COHORT = False
RUN_EVALUATE_R12 = False

targets = DEVELOPMENT_TARGETS + (PROSPECTIVE_TARGETS if RUN_PROSPECTIVE_COHORT else [])

for subject in targets:
    for extra_flags in ([], ["--scratch"]):
        subprocess.run([
            sys.executable, "-m", "eeg_decoder.gpu_pipeline", "adapt",
            "--config", CONFIG, "--work-dir", WORK_DIR, "--device", DEVICE,
            "--subject", str(subject), "--resume", *extra_flags,
        ], check=True)
        subprocess.run([
            sys.executable, "-m", "eeg_decoder.gpu_pipeline", "calibrate",
            "--config", CONFIG, "--work-dir", WORK_DIR, "--device", DEVICE,
            "--subject", str(subject), *extra_flags,
        ], check=True)
        if RUN_EVALUATE_R12:
            subprocess.run([
                sys.executable, "-m", "eeg_decoder.gpu_pipeline", "evaluate",
                "--config", CONFIG, "--work-dir", WORK_DIR, "--device", DEVICE,
                "--subject", str(subject), *extra_flags,
            ], check=True)


In [ ]:
# [Step 4] Inspect Calibrated Thresholds and Optional GitHub Sync
import json
import pathlib

work = pathlib.Path(WORK_DIR)
for kind in ("transfer", "scratch"):
    for cal_file in sorted((work / kind).glob("S*/calibration.json")):
        cal = json.loads(cal_file.read_text(encoding="utf-8"))
        print(
            f"{kind:8s} | S{cal['subject']:03d} | threshold={cal['threshold']:.3f} "
            f"| status={cal['status']} | crc_adj_risk={cal['selected_crc_adjusted_risk']}"
        )
